In [1]:
# Scaled dot- product attention
import numpy as np

# step-1 computing the dot product of query and keys
def compute_scores(Q, K):
    return np.dot(Q, K.T)

In [2]:
# Step -2 Scale the scores by the square root of the dimension of keys

def scale_scores(scores, d_k):
    return scores / np.sqrt(d_k)

In [3]:
# step-3 Applying softmax function to the scores
def softmax(scores):
  exp_scores = np.exp(scores - np.max(scores))
  return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

In [4]:
# Step-4 Compute the weighted sum of the values

def compute_attention_output(attention_weights, V):
  return np.dot(attention_weights, V)

In [5]:
def attention(Q, K, V):
  d_k = Q.shape[-1]
  scores = compute_scores(Q, K)
  scaled_scores = scale_scores(scores, d_k)
  attention_weights = softmax(scaled_scores)
  output = compute_attention_output(attention_weights, V)

  return output, attention_weights

In [6]:
# Example input: Q, K, V

Q = np.array([[1, 0]])
K = np.array([[1, 0], [0, 1], [1,1]])
V = np.array([[1, 0], [0, 1], [1,0]])

# Apply attention
output, attention_weights = attention(Q, K, V)
print("Output:", output)
print("Attention Weights:", attention_weights)

Output: [[0.80222419 0.19777581]]
Attention Weights: [[0.40111209 0.19777581 0.40111209]]


## Text Classification

In [8]:
import numpy as np
import tensorflow as tf

sentences = [
    'I love programming',
    'I hate bugs',
    'bugs are annoying',
    'programming is cool',
    'I enjoy programming',
    'I despire errors',
    'errors make me sad',
    'coding makes me happy'
]
labels = [1, 0, 0, 1, 1, 0, 0, 1] # positive -1, negative -0

# Build vocabulory

tokenizer = tf.keras.preprocessing.text.Tokenizer(lower = True, oov_token='<OOV>')
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index
vocab_size = len(word_index) +1

In [38]:
# convert sequences and add padding

sequences = tokenizer.texts_to_sequences(sentences)
x = tf.keras.preprocessing.sequence.pad_sequences(sequences, padding='post')
y= np.array(labels)

In [39]:
# Parameters

embedding_dim = 16
hidden_dim = 16
output_dim = 2
maxlen = x.shape[1]

In [40]:
# Self attention layer
class SelfAttention(tf.keras.layers.Layer):
    def __init__(self, hidden_dim):
        super(SelfAttention, self).__init__()
        self.W_q = tf.keras.layers.Dense(hidden_dim)
        self.W_k = tf.keras.layers.Dense(hidden_dim)
        self.W_v = tf.keras.layers.Dense(hidden_dim)

    def call(self, inputs):
        Q = self.W_q(inputs)
        K = self.W_k(inputs)
        V = self.W_v(inputs)

        scores = tf.matmul(Q, K, transpose_b=True)
        dim_k = tf.cast(tf.shape(K)[-1], tf.float32)
        scaled_scores = scores / tf.math.sqrt(dim_k)
        attention_weights = tf.nn.softmax(scaled_scores, axis=-1)
        context= tf.matmul(attention_weights, V)

        return context

In [41]:
# Build the model

inputs = tf.keras.Input(shape=(maxlen, ))
x = tf.keras.layers.Embedding(vocab_size, embedding_dim)(inputs)
x = SelfAttention(hidden_dim)(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dense(16, activation='relu')(x)

outputs = tf.keras.layers.Dense(output_dim, activation='softmax')(x)


In [42]:
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=30, verbose=1)

Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5000 - loss: 0.6948
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.6250 - loss: 0.6924
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.6250 - loss: 0.6901
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.8750 - loss: 0.6878
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.8750 - loss: 0.6858
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8750 - loss: 0.6839
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 1.0000 - loss: 0.6821
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 1.0000 - loss: 0.6802
Epoch 9/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 1.0000 - loss: 0.6782
Epoch 10/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 1.0000 - loss: 0.6762
Epoch 11/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 1.0000 - loss: 0.6741
Epoch 12/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 1.0000 - loss: 0.6718
Epo

In [43]:
# Predictions

def predict_sentiment(text):
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = tf.keras.preprocessing.sequence.pad_sequences(sequence, maxlen=maxlen, padding='post')
    prediction = model.predict(padded_sequence)[0]
    label = np.argmax(prediction)
    sentiment = 'positive' if label == 1 else 'negative'
    print(f"Input- {text}")
    print(f"prediction - {sentiment} | probabilities - {prediction}")


In [44]:
# Test

predict_sentiment("I love programming")
predict_sentiment("I hate bugs")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
Input- I love programming
prediction - positive | probabilities - [0.4632718 0.5367282]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
Input- I hate bugs
prediction - negative | probabilities - [0.5452442 0.4547558]


In [45]:
predict_sentiment("bugs are annoying")
predict_sentiment("programming is cool")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Input- bugs are annoying
prediction - negative | probabilities - [0.54780537 0.4521946 ]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
Input- programming is cool
prediction - positive | probabilities - [0.444095   0.55590504]
